In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, cross_val_score
import os
import sys
current_dir = os.path.dirname(os.path.abspath(__name__))
sys.path.append(os.path.join(current_dir, '..', 'src'))
from feature import preprocess_titanic # 确保文件名一致

# A. 加载数据
train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
passenger_id = test['PassengerId']

# B. 应用特征工程
train_df = preprocess_titanic(train)
test_df = preprocess_titanic(test)

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

# C. 自动调参 (Grid Search)
# 这是冲刺 0.80 的秘密武器：让计算机帮你找最佳参数
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 7, 9],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

rf = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X, y)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Score: {grid_search.best_score_:.4f}")

# D. 评估 Loss (使用 Log Loss 观察模型置信度)
from sklearn.metrics import log_loss
val_probs = grid_search.predict_proba(X)
print(f"Training Log Loss: {log_loss(y, val_probs):.4f}")

Best Parameters: {'criterion': 'entropy', 'max_depth': 9, 'min_samples_leaf': 4, 'n_estimators': 200}
Best CV Score: 0.8328
Training Log Loss: 0.3064


In [6]:
best_model = grid_search.best_estimator_
predictions = best_model.predict(test_df)

# 生成 CSV
submission = pd.DataFrame({
    "PassengerId": passenger_id,
    "Survived": predictions
})

submission.to_csv('../submissions/submission_rf_v2.csv', index=False)